# Model A — Phase 6: risk calculator (simplified score) and final reporting
### Set 2 (raw), restricted model (N=30), under both outcome definitions — unadjusted and adjusted

This phase builds the risk calculator (the gradient boosting model restricted to the most important
variables, not a manual point score) for Set 2 — the set with discrimination superior to SAPS3, sustained
under both outcome definitions (p<0.001 both unadjusted and adjusted). Even so, this phase builds and
validates the calculator under both definitions side by side, rather than choosing one in advance: the
DeLong-based selection of the optimal N did not converge cleanly under the adjusted definition (the
smallest N statistically equivalent to the full model differed between the two definitions, and the final
N=30 decision rested partly on visual reading of the ablation curve, not only on the statistical test),
and the calculator's discrimination margin against the full model is also narrower under the adjusted
definition. Validating both definitions side by side avoids committing to an N or outcome choice before
these asymmetries are fully resolved.

**What this phase does:**
1. Loads Set 2's full model already fitted under both definitions (unadjusted — saved in Phase 4;
   adjusted — saved in Phase 5, Part II, Section 6), instead of rebuilding the cohort and retraining
   here again.
2. Recomputes the SHAP importance ranking separately for each definition.
3. Selects the top-N (30) variables (the N already decided by DeLong, shared by both definitions) in
   each definition, and checks whether the list changes.
4. Fits the calculator (restricted model) for each definition, and validates its discrimination against
   the full model (ROC, DeLong).
5. Fully validates the calculator, replicating Phase 5's methodology: classification metrics at three
   thresholds (Youden, ≈0.10, ≈0.20) and post-hoc recalibration (bootstrap decision + out-of-fold Platt
   scaling), for both definitions.
6. Packages the calculator as a simple Python function, for each definition.
7. Fills in the TRIPOD-AI checklist and the PROBAST risk-of-bias assessment.

## 0 · Configuration and loading

> **Reproducibility note.** The original study ran in Google Colab with the source spreadsheet on Google Drive. In this repository, place the input files in `data/` and adjust the paths accordingly.

In [ ]:
!pip -q install shap --upgrade

In [ ]:
import os, json
import numpy as np, pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import joblib
import shap
from sklearn.base import clone
from sklearn.metrics import (roc_auc_score, roc_curve, brier_score_loss, confusion_matrix,
                              accuracy_score, f1_score, matthews_corrcoef)
from sklearn.calibration import calibration_curve
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedGroupKFold
pd.set_option('display.max_columns', 60)

# === Configuration ===
FILE_PATH = 'data/Excel01GlobalAdmission.xlsx'
FASE2_DIR = 'outputs/phase2'
FASE4_DIR = 'outputs/phase4'
FASE5_DIR = 'outputs/phase5_adjusted'  # Phase 5 Part II — saved artifact (Set 2, adjusted)
OUTDIR    = 'outputs/phase6'
os.makedirs(OUTDIR, exist_ok=True)

CSV_PATH  = f'{FASE2_DIR}/modelo_A_fase2.csv'
CONJ_PATH = f'{FASE2_DIR}/modelo_A_sets_variaveis.json'

MERGE_KEYS = ['EpimedCode', 'MedicalRecord']
CORTE_TEMPORAL = 2023
ALVO = 'mortalidade_hospitalar'
SEED = 42
N_TOP_DIAG = 20
N_POSICOES = 12
N_TOP = 30  # decided by DeLong: smallest N with no significant difference vs. the full model, valid both unadjusted (p=0.671) and adjusted (p=0.053)
NOME_CONJUNTO = 'Conjunto 2 (brutos)'

CATEGORICAS = ['Gender','AdmissionSourceName','AdmissionTypeName','AdmissionReasonName',
               'AdmissionMainDiagnosisName_grp','Chronic Health Status Name','Status COVID 19',
               'OrigemPagamento']

print('Configuration loaded.')

In [ ]:
def preparar_boosting(set, treino, teste):
    cats = [c for c in set if c in CATEGORICAS]
    nums = [c for c in set if c not in CATEGORICAS and c != 'Month Year']
    cols = cats + nums
    Xtr = treino[cols].copy()
    Xte = teste[cols].copy()
    for c in cats:
        Xtr[c] = Xtr[c].astype('object').fillna('Not reported')
        Xte[c] = Xte[c].astype('object').fillna('Not reported')
        categorias = pd.Index(Xtr[c].unique()).union(['Not reported'])
        Xte[c] = Xte[c].where(Xte[c].isin(categorias), 'Not reported')
        Xtr[c] = pd.Categorical(Xtr[c], categories=categorias)
        Xte[c] = pd.Categorical(Xte[c], categories=categorias)
    return Xtr, Xte, cats, nums

def agrupar_diagnostico_principal(treino, teste, col='AdmissionMainDiagnosisName', n_top=N_TOP_DIAG):
    col_grp = f'{col}_grp'
    top_categorias = treino[col].value_counts().nlargest(n_top).index
    treino = treino.copy(); teste = teste.copy()
    treino[col_grp] = treino[col].where(treino[col].isin(top_categorias), 'outros')
    teste[col_grp]  = teste[col].where(teste[col].isin(top_categorias), 'outros')
    return treino, teste

def usar_diagnostico_agrupado(set):
    return ['AdmissionMainDiagnosisName_grp' if c == 'AdmissionMainDiagnosisName' else c for c in set]

with open(CONJ_PATH, encoding='utf-8') as f:
    sets = json.load(f)
CONJUNTO_2 = usar_diagnostico_agrupado(sets['set_2_bruto'])
BENCHMARK = sets['benchmark']

print('Helper functions defined.')

## 1 · Full model — unadjusted definition (already trained in Phase 4)

In [ ]:
df = pd.read_csv(CSV_PATH)
if 'ano' not in df.columns:
    df['ano'] = pd.to_datetime(df['Month Year'], format='%m/%Y').dt.year

treino_sem_ajuste = df[df['ano'] <  CORTE_TEMPORAL].copy()
teste_sem_ajuste  = df[df['ano'] >= CORTE_TEMPORAL].copy()
treino_sem_ajuste, teste_sem_ajuste = agrupar_diagnostico_principal(treino_sem_ajuste, teste_sem_ajuste)

Xtr_sem_ajuste, Xte_sem_ajuste, cats_sem_ajuste, nums_sem_ajuste = preparar_boosting(CONJUNTO_2, treino_sem_ajuste, teste_sem_ajuste)
ytr_sem_ajuste = treino_sem_ajuste[ALVO].values
yte_sem_ajuste = teste_sem_ajuste[ALVO].values

modelo_completo_sem_ajuste = joblib.load(f'{FASE4_DIR}/modelo_2_XGBoost.joblib')
p_teste_completo_sem_ajuste = modelo_completo_sem_ajuste.predict_proba(Xte_sem_ajuste)[:, 1]
auroc_completo_sem_ajuste = roc_auc_score(yte_sem_ajuste, p_teste_completo_sem_ajuste)

print(f'Full model (unadjusted): n_variables={Xtr_sem_ajuste.shape[1]}  AUROC={auroc_completo_sem_ajuste:.4f}')

## 2 · Full model — adjusted definition (loaded from Phase 5)

Set 2's full model, retrained on the cohort with the adjusted definition (transferred patients excluded),
was already fitted and saved in Phase 5 (Part II, Section 6 — "Refitting the winning engines on the
adjusted cohort").

In [ ]:
# --- Load the artifact saved in Phase 5 (Part II, Section 6): model, data, and predictions, all ready ---
ARTEFATO_FASE5_PATH = f'{FASE5_DIR}/modelo_completo_com_ajuste_set2.joblib'

artefato_com_ajuste = joblib.load(ARTEFATO_FASE5_PATH)

modelo_completo_com_ajuste = artefato_com_ajuste['modelo']
Xtr_com_ajuste = artefato_com_ajuste['Xtr']
Xte_com_ajuste = artefato_com_ajuste['Xte']
ytr_com_ajuste = artefato_com_ajuste['ytr']
yte_com_ajuste = artefato_com_ajuste['yte']
cats_com_ajuste = artefato_com_ajuste['cats']
groups_tr_com_ajuste = artefato_com_ajuste['groups_tr']
p_teste_completo_com_ajuste = artefato_com_ajuste['p_teste']
patient_ids_teste_com_ajuste = artefato_com_ajuste['patient_ids_teste']

auroc_completo_com_ajuste = roc_auc_score(yte_com_ajuste, p_teste_completo_com_ajuste)

print(f'Modelo completo (com ajuste) carregado da Fase 5: n_variaveis={Xtr_com_ajuste.shape[1]}  '
      f'AUROC={auroc_completo_com_ajuste:.4f}  |  n_treino={len(ytr_com_ajuste)}  n_teste={len(yte_com_ajuste)}')

## 3 · SHAP importance ranking — each definition, separately

In [ ]:
def ranking_shap(modelo, Xtr):
    explainer = shap.TreeExplainer(modelo)
    shap_vals = explainer(Xtr)
    importancia = pd.Series(np.abs(shap_vals.values).mean(axis=0), index=Xtr.columns)
    return importancia.sort_values(ascending=False)

importancia_sem_ajuste = ranking_shap(modelo_completo_sem_ajuste, Xtr_sem_ajuste)
importancia_com_ajuste  = ranking_shap(modelo_completo_com_ajuste, Xtr_com_ajuste)

ranking_sem_ajuste = importancia_sem_ajuste.index.tolist()
ranking_com_ajuste  = importancia_com_ajuste.index.tolist()

print('Top 30 (unadjusted):', ranking_sem_ajuste[:30])
print('Top 30 (com ajuste):', ranking_com_ajuste[:30])

In [ ]:
NOME_SEM_AJUSTE = 'Sem ajuste'
NOME_COM_AJUSTE = 'Com ajuste'
N_EXIBIR = 30

top_sem_ajuste = importancia_sem_ajuste.head(N_EXIBIR)
top_com_ajuste = importancia_com_ajuste.head(N_EXIBIR)
variaveis_comuns = set(top_sem_ajuste.index) & set(top_com_ajuste.index)

fig, axes = plt.subplots(1, 2, figsize=(13, 7))

for eixo, top, nome_def, outro_top in [
    (axes[0], top_sem_ajuste, NOME_SEM_AJUSTE, top_com_ajuste),
    (axes[1], top_com_ajuste, NOME_COM_AJUSTE, top_sem_ajuste),
]:
    cores = ['#1F3864' if v in variaveis_comuns else '#C00000' for v in top.index]
    y_pos = np.arange(len(top))
    eixo.barh(y_pos, top.values, color=cores)
    eixo.set_yticks(y_pos)
    eixo.set_yticklabels(top.index, fontsize=9)
    eixo.invert_yaxis()
    eixo.set_xlabel('Mean |SHAP|')
    eixo.set_title(f'{nome_def} — top {N_EXIBIR}', fontsize=12, fontweight='bold')

legenda_cores = [
    plt.matplotlib.patches.Patch(facecolor='#1F3864', label=f'Presente nas duas listas (top {N_EXIBIR})'),
    plt.matplotlib.patches.Patch(facecolor='#C00000', label='Only in this definition'),
]
fig.legend(handles=legenda_cores, loc='lower center', ncol=2, bbox_to_anchor=(0.5, -0.05), frameon=False, fontsize=9)
fig.suptitle('SHAP importance — without vs. with transfer-bias adjustment', fontsize=13, fontweight='bold', y=1.02)

plt.tight_layout()
fig_shap_comp_path = f'{OUTDIR}/fase6_shap_sem_ajuste_vs_com_ajuste.png'
fig.savefig(fig_shap_comp_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Figura salva: {fig_shap_comp_path}')

## 4 · Select the top-N (30) variables — compare the two lists

In [ ]:
variaveis_top_sem_ajuste = ranking_sem_ajuste[:N_TOP]
variaveis_top_com_ajuste  = ranking_com_ajuste[:N_TOP]

em_comum = set(variaveis_top_sem_ajuste) & set(variaveis_top_com_ajuste)
so_sem_ajuste = set(variaveis_top_sem_ajuste) - set(variaveis_top_com_ajuste)
so_com_ajuste  = set(variaveis_top_com_ajuste) - set(variaveis_top_sem_ajuste)

print(f'Variables common to both top-{N_TOP} lists: {len(em_comum)} of {N_TOP}')
print(f'Only in the unadjusted list: {sorted(so_sem_ajuste)}')
print(f'Only in the adjusted list:  {sorted(so_com_ajuste)}')

In [ ]:
# --- Full table: each definition's N_TOP variables, side by side, with ranking position ---
tab_topN = pd.DataFrame({
    'posicao': range(1, N_TOP + 1),
    'Sem ajuste': variaveis_top_sem_ajuste,
    'Com ajuste': variaveis_top_com_ajuste,
})
tab_topN['mesma_variavel_na_posicao'] = tab_topN['Sem ajuste'] == tab_topN['Com ajuste']

print(tab_topN.to_string(index=False))

topN_path = f'{OUTDIR}/fase6_top{N_TOP}_variaveis_comparacao.csv'
tab_topN.to_csv(topN_path, index=False, encoding='utf-8-sig')
print(f'\nTabela salva: {topN_path}')

In [ ]:
# --- Plot: Venn diagram of the two top-N lists, with the full list of shared variables alongside ---
from matplotlib.patches import Circle

fig, (ax_venn, ax_lista) = plt.subplots(1, 2, figsize=(15, 6.5), gridspec_kw={'width_ratios': [1.1, 1]})

raio = 2.2
centro_esq = (-0.9, 0)
centro_dir = (0.9, 0)

circ_esq = Circle(centro_esq, raio, color='#8FAADC', alpha=0.5, ec='#1F3864', linewidth=1.5)
circ_dir = Circle(centro_dir, raio, color='#F4B183', alpha=0.5, ec='#C00000', linewidth=1.5)
ax_venn.add_patch(circ_esq)
ax_venn.add_patch(circ_dir)

ax_venn.text(centro_esq[0] - 1.0, 0, f'{len(so_sem_ajuste)}', fontsize=22, fontweight='bold', ha='center', va='center', color='#1F3864')
ax_venn.text(0, 0, f'{len(em_comum)}', fontsize=26, fontweight='bold', ha='center', va='center', color='#333333')
ax_venn.text(centro_dir[0] + 1.0, 0, f'{len(so_com_ajuste)}', fontsize=22, fontweight='bold', ha='center', va='center', color='#C00000')

ax_venn.text(centro_esq[0], raio + 0.4, 'Sem ajuste', fontsize=13, fontweight='bold', ha='center', color='#1F3864')
ax_venn.text(centro_dir[0], raio + 0.4, 'Com ajuste', fontsize=13, fontweight='bold', ha='center', color='#C00000')

texto_so_sem_ajuste = '\n'.join(sorted(so_sem_ajuste)) if so_sem_ajuste else '(nenhuma)'
texto_so_com_ajuste  = '\n'.join(sorted(so_com_ajuste)) if so_com_ajuste else '(nenhuma)'
ax_venn.text(centro_esq[0] - 1.0, -raio - 0.5, texto_so_sem_ajuste, fontsize=9, ha='center', va='top', color='#1F3864')
ax_venn.text(centro_dir[0] + 1.0, -raio - 0.5, texto_so_com_ajuste, fontsize=9, ha='center', va='top', color='#C00000')

ax_venn.set_xlim(-5, 5)
ax_venn.set_ylim(-4.5, 3.5)
ax_venn.set_aspect('equal')
ax_venn.axis('off')
ax_venn.set_title(f'Top-{N_TOP} variables (SHAP)\noverlap between the two definitions', fontsize=12, fontweight='bold')

# --- Text panel: full list of shared variables ---
lista_comum_ordenada = sorted(em_comum)
texto_lista = '\n'.join(f'{i+1:>2}. {v}' for i, v in enumerate(lista_comum_ordenada))
ax_lista.axis('off')
ax_lista.text(0, 1, f'Variables in common ({len(em_comum)} of {N_TOP}):', fontsize=12, fontweight='bold',
              ha='left', va='top', transform=ax_lista.transAxes, color='#333333')
ax_lista.text(0, 0.92, texto_lista, fontsize=9.5, ha='left', va='top', transform=ax_lista.transAxes,
              family='monospace', color='#1F3864')

plt.tight_layout()
fig_venn_path = f'{OUTDIR}/fase6_venn_top{N_TOP}.png'
fig.savefig(fig_venn_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Figura salva: {fig_venn_path}')

## 5 · Fit the calculator (restricted model) — each definition

### Note — what the calculator is, technically

The "calculator" is not a new model trained from scratch: it is the same gradient boosting engine as the
full model (same hyperparameters, via `clone()`), refitted using **only the N=30 variables** with highest
SHAP importance in each definition — the weights learned by the full model (132 variables) are not
reused; the model is retrained from scratch, just with fewer input columns. That is why it is called a
"restricted model", not a "point score" — there is no per-variable point sum, as in a traditional
clinical score (e.g., SAPS3); the output remains a continuous probability, produced by the same kind of
decision trees as the original model.

Each outcome definition has its own calculator, with its own set of 30 variables
(`variaveis_top_sem_ajuste` and `variaveis_top_com_ajuste`, decided in Section 4) — the lists overlap
heavily but are not identical. `cats_top` filters, within those 30, which are categorical, so the engine
treats them correctly.

**Reading this cell's numbers**: the calculator's AUROC and Brier, side by side with the full model's,
per definition. The AUROC difference (unadjusted: 0.9067→0.9056; adjusted: 0.9070→0.9007) already
anticipates what Section 6 confirms with a formal test (DeLong): the discrimination loss from reducing
to 30 variables is small and non-significant unadjusted, but larger and statistically significant
adjusted. The Brier score (mean squared error of the predicted probability) is reported here only as
accompaniment — the proper calibration validation (recalibration decision, before/after curves) comes in
Section 8.

In [ ]:
def ajustar_calculadora(modelo_completo, Xtr, ytr, cats, variaveis_top):
    cats_top = [c for c in cats if c in variaveis_top]
    Xtr_top = Xtr[variaveis_top]
    modelo_restrito = clone(modelo_completo)
    if 'LGBM' in type(modelo_restrito).__name__:
        modelo_restrito.fit(Xtr_top, ytr, categorical_feature=cats_top)
    else:
        modelo_restrito.fit(Xtr_top, ytr)
    return modelo_restrito

calculadora_sem_ajuste = ajustar_calculadora(modelo_completo_sem_ajuste, Xtr_sem_ajuste, ytr_sem_ajuste,
                                            cats_sem_ajuste, variaveis_top_sem_ajuste)
calculadora_com_ajuste = ajustar_calculadora(modelo_completo_com_ajuste, Xtr_com_ajuste, ytr_com_ajuste,
                                            cats_com_ajuste, variaveis_top_com_ajuste)

p_teste_calculadora_sem_ajuste = calculadora_sem_ajuste.predict_proba(Xte_sem_ajuste[variaveis_top_sem_ajuste])[:, 1]
p_teste_calculadora_com_ajuste = calculadora_com_ajuste.predict_proba(Xte_com_ajuste[variaveis_top_com_ajuste])[:, 1]

print(f'Calculator (unadjusted): AUROC = {roc_auc_score(yte_sem_ajuste, p_teste_calculadora_sem_ajuste):.4f}  '
      f'(modelo completo: {auroc_completo_sem_ajuste:.4f})  |  '
      f'Brier = {brier_score_loss(yte_sem_ajuste, p_teste_calculadora_sem_ajuste):.4f}')
print(f'Calculadora (com ajuste): AUROC = {roc_auc_score(yte_com_ajuste, p_teste_calculadora_com_ajuste):.4f}  '
      f'(modelo completo: {auroc_completo_com_ajuste:.4f})  |  '
      f'Brier = {brier_score_loss(yte_com_ajuste, p_teste_calculadora_com_ajuste):.4f}')

## 6 · Validate — full vs. calculator ROC, and the DeLong test, per definition

In [ ]:
def _compute_midrank(x):
    J = np.argsort(x); Z = x[J]; N = len(x)
    T = np.zeros(N, dtype=float)
    i = 0
    while i < N:
        j = i
        while j < N and Z[j] == Z[i]:
            j += 1
        T[i:j] = 0.5 * (i + j - 1) + 1
        i = j
    T2 = np.empty(N, dtype=float); T2[J] = T
    return T2

def _fast_delong(preds_sorted_transposed, m):
    k = preds_sorted_transposed.shape[0]
    n = preds_sorted_transposed.shape[1] - m
    tx = np.empty([k, m]); ty = np.empty([k, n]); tz = np.empty([k, m + n])
    for r in range(k):
        tx[r, :] = _compute_midrank(preds_sorted_transposed[r, :m])
        ty[r, :] = _compute_midrank(preds_sorted_transposed[r, m:])
        tz[r, :] = _compute_midrank(preds_sorted_transposed[r, :])
    aucs = tz[:, :m].sum(axis=1) / m / n - float(m + 1.0) / 2.0 / n
    v01 = (tz[:, :m] - tx[:, :]) / n
    v10 = 1.0 - (tz[:, m:] - ty[:, :]) / m
    sx = np.cov(v01); sy = np.cov(v10)
    return aucs, sx / m + sy / n

def teste_delong_duas_curvas(y_true, p1, p2):
    from scipy import stats as sstats
    order = np.argsort(-y_true)
    y_ord = y_true[order]
    m = int(y_ord.sum())
    preds = np.vstack([p1[order], p2[order]])
    aucs, cov = _fast_delong(preds, m)
    diff = aucs[0] - aucs[1]
    var = cov[0, 0] + cov[1, 1] - 2 * cov[0, 1]
    z = diff / np.sqrt(var) if var > 0 else np.nan
    p_valor = 2 * (1 - sstats.norm.cdf(abs(z))) if var > 0 else np.nan
    return diff, p_valor

diff_sem_ajuste, p_valor_sem_ajuste = teste_delong_duas_curvas(yte_sem_ajuste, p_teste_calculadora_sem_ajuste, p_teste_completo_sem_ajuste)
diff_com_ajuste, p_valor_com_ajuste = teste_delong_duas_curvas(yte_com_ajuste, p_teste_calculadora_com_ajuste, p_teste_completo_com_ajuste)

print(f'Unadjusted: calculator vs. full — difference={diff_sem_ajuste:+.4f}  p={p_valor_sem_ajuste:.3f}')
print(f'Adjusted: calculator vs. full — difference={diff_com_ajuste:+.4f}  p={p_valor_com_ajuste:.3f}')

In [ ]:
# --- Plot: full vs. calculator ROC, for both definitions, side by side ---
fig, axes = plt.subplots(1, 2, figsize=(13, 6))

configuracoes = [
    (axes[0], 'Sem ajuste', yte_sem_ajuste, p_teste_completo_sem_ajuste, p_teste_calculadora_sem_ajuste,
     auroc_completo_sem_ajuste, roc_auc_score(yte_sem_ajuste, p_teste_calculadora_sem_ajuste), p_valor_sem_ajuste,
     Xtr_sem_ajuste.shape[1]),
    (axes[1], 'Com ajuste', yte_com_ajuste, p_teste_completo_com_ajuste, p_teste_calculadora_com_ajuste,
     auroc_completo_com_ajuste, roc_auc_score(yte_com_ajuste, p_teste_calculadora_com_ajuste), p_valor_com_ajuste,
     Xtr_com_ajuste.shape[1]),
]

for eixo, nome_def, y_true, p_completo, p_calc, auroc_completo, auroc_calc, p_valor, n_completo in configuracoes:
    fpr_c, tpr_c, _ = roc_curve(y_true, p_completo)
    fpr_r, tpr_r, _ = roc_curve(y_true, p_calc)
    eixo.plot(fpr_c, tpr_c, color='#1F3864', linewidth=2.2, label=f'Completo (n={n_completo})  AUROC={auroc_completo:.3f}')
    eixo.plot(fpr_r, tpr_r, color='#C00000', linewidth=2.2, linestyle='--', label=f'Calculadora (n={N_TOP})  AUROC={auroc_calc:.3f}')
    eixo.plot([0, 1], [0, 1], color='gray', linewidth=1, linestyle=':')
    eixo.set_xlabel('1 − especificidade (FPR)')
    eixo.set_ylabel('Sensibilidade (TPR)')
    eixo.set_title(f'{nome_def}\nDeLong calculadora vs. completo: p={p_valor:.3f}')
    eixo.legend(loc='lower right', fontsize=9)

plt.tight_layout()
fig_calc_path = f'{OUTDIR}/fase6_roc_calculadora_completo_sem_ajuste_vs_com_ajuste.png'
fig.savefig(fig_calc_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Figura salva: {fig_calc_path}')

### Note — reading the ROC curves (calculator vs. full model, per definition)

The two curves nearly overlap visually in both panels, but the DeLong test shows that this visual
similarity does not mean the same thing in the two definitions.

**Unadjusted**: AUROC 0.907 (full) vs. 0.906 (calculator), p=0.671. The difference is not statistically
significant — reducing from 132 to 30 variables causes no detectable discrimination loss under this
definition.

**Adjusted**: AUROC 0.907 (full) vs. 0.901 (calculator), p=0.014. Here the difference **is** statistically
significant, despite the curves looking nearly identical — the same pattern already seen in Phase 5
(Part II) for this same N=30. That is, under the adjusted definition, the reduction to 30 variables has a
real discrimination cost, small in magnitude (−0.006 AUROC) but not attributable to sampling noise.

**Practical implication.** The calculator (N=30) is a statistically safe simplification under the
unadjusted definition, but not under the adjusted one — there, the full model (132 variables) preserves
discrimination that the simplified version measurably loses. This does not invalidate the calculator,
but limits the equivalence claim against the full model to only one of the two outcome definitions.

In [ ]:
print('n_test (unadjusted):', len(yte_sem_ajuste))
print('n_teste (com ajuste):', len(yte_com_ajuste))

# Variance/covariance behind DeLong, to see whether the explanation is "more correlation" or "a larger gap"
for nome, y, p1, p2 in [('Sem ajuste', yte_sem_ajuste, p_teste_calculadora_sem_ajuste, p_teste_completo_sem_ajuste),
                         ('Com ajuste', yte_com_ajuste, p_teste_calculadora_com_ajuste, p_teste_completo_com_ajuste)]:
    order = np.argsort(-y)
    m = int(y[order].sum())
    preds = np.vstack([p1[order], p2[order]])
    aucs, cov = _fast_delong(preds, m)
    print(f'{nome}: var_completo={cov[1,1]:.6f}  var_calc={cov[0,0]:.6f}  cov={cov[0,1]:.6f}  '
          f'var_diferenca={cov[0,0]+cov[1,1]-2*cov[0,1]:.6f}')

### Note — what this cell is checking

The DeLong test compares two paired AUROCs (same patients, two predictions). The significance of this
test depends on two things, not one: the size of the difference between the AUROCs, and the variance of
that difference — which, in turn, depends on how correlated the two predictions are. Since the
calculator is a refitted version of the same engine, using a subset of the full model's variables, the
two predictions are expected to be highly correlated (high `cov`) — which reduces the variance of the
difference and makes the test more sensitive to small differences.

**Why this matters here.** The "adjusted" definition gave p=0.014 (significant) and the "unadjusted" gave
p=0.671 (non-significant), despite both having a small AUROC drop (−0.0011 and −0.0051, respectively).
This cell exists to answer: does that significance difference come from a smaller variance under the
adjusted definition (more correlation between full model and calculator, making the test more
"sensitive") — or does it simply come from the AUROC drop being larger there?

**Reading the numbers.** `var_diferenca` is practically identical across the two definitions (0.000007 in
both). That is, the paired test's variance **does not change** between the two conditions — the
correlation between calculator and full model is the same in both cases. This rules out the first
hypothesis: it is not "more statistical sensitivity" that makes the adjusted definition significant. The
explanation is the second — the real AUROC drop is larger there (−0.0051 vs. −0.0011), and since the
test's "thermometer" (the variance) is the same in both definitions, that difference in magnitude is
enough, by itself, to cross the significance threshold in one definition and not the other.

In [ ]:
# --- Isolate the test-composition effect: "unadjusted" models evaluated on the same patient subset ---
# patient_ids_teste_com_ajuste comes from the artifact loaded from Phase 5 (Section 2) — these are exactly the patients
# of the test under the adjusted definition (i.e., the non-transferred); no need to rebuild the flag here.
mascara_nao_transferido = teste_sem_ajuste['patient_id'].isin(set(patient_ids_teste_com_ajuste)).values

yte_sem_ajuste_matched = yte_sem_ajuste[mascara_nao_transferido]
p_completo_matched = p_teste_completo_sem_ajuste[mascara_nao_transferido]
p_calc_matched = p_teste_calculadora_sem_ajuste[mascara_nao_transferido]

print('n (deveria bater com 1.277):', mascara_nao_transferido.sum())
print('Full-model AUROC (unadjusted, non-transferred patients only):', roc_auc_score(yte_sem_ajuste_matched, p_completo_matched))
print('Calculator AUROC (unadjusted, non-transferred patients only):', roc_auc_score(yte_sem_ajuste_matched, p_calc_matched))

diff_matched, p_matched = teste_delong_duas_curvas(yte_sem_ajuste_matched, p_calc_matched, p_completo_matched)
print(f'Difference (matched): {diff_matched:+.4f}  p={p_matched:.3f}')

### Note — why re-evaluate the "unadjusted" model on a smaller patient subset

Section 6's DeLong test compared calculator vs. full model on two test sets of different sizes:
1,400 patients (unadjusted) and 1,277 (adjusted) — transferred patients leave the test set only under
the adjusted definition. This raises a legitimate doubt: does the significance difference between the two
definitions (p=0.671 vs. p=0.014, Section 6) come from the outcome definition itself, or simply from the
two test sets having different sizes and compositions?

**What this cell does.** It takes the same "unadjusted" models (full and calculator) and re-evaluates
their predictions, but restricting the test to the **same 1,277 patients** composing the "adjusted" test
set (via `patient_ids_teste_com_ajuste`, loaded from Phase 5). That is: same outcome definition
(unadjusted), same patient composition the adjusted definition uses. This isolates the effect of "how
many and which patients are in the test", holding the outcome definition fixed.

**Reading the numbers.** `n=1,277` matches the "adjusted" test exactly — the filter is picking the right
patient set. Full-model AUROC (0.9070) and calculator AUROC (0.9054) on this subset sit very close to the
original values on 1,400 patients (0.9067 and 0.9056, Section 5) — that is, shrinking the test from 1,400
to 1,277 patients, by itself, barely changes the numbers. The (matched) difference is −0.0015, p=0.555:
still non-significant, and a much smaller difference than observed under the adjusted definition
(−0.0051, p=0.014).

**Conclusion.** Test size/composition is not the cause of the significance difference between the two
definitions — when the unadjusted definition is held fixed and only the patient subset changes, the
calculator's discrimination remains statistically equivalent to the full model's. The statistically
significant discrimination loss seen under the adjusted definition is therefore attributable to the
outcome change itself (excluding transferred patients), not to a sampling artifact.

## 7 · Classification metrics at operational thresholds — calculator

Same methodology as Phase 5 (Part I, Section 2), now applied to the calculator (restricted model, N=30),
under both outcome definitions. Three thresholds: the Youden index — computed from the calculator's own
ROC curve in each definition, and so may differ between them — and two fixed clinical thresholds (0.10
and 0.20), in the same positive net-benefit range mapped in Phase 4. Accuracy and F1 are reported for
completeness; MCC is the preferred scalar summary, for the same reasons recorded in Phase 5
(Chicco & Jurman, 2020).

In [ ]:
def limiar_youden(y, p):
    fpr, tpr, limiares = roc_curve(y, p)
    j = tpr - fpr
    return limiares[np.argmax(j)]


LIMIARES_CLINICOS = [0.10, 0.20]

predicoes_calculadora = {
    'Sem ajuste': (yte_sem_ajuste, p_teste_calculadora_sem_ajuste),
    'Com ajuste': (yte_com_ajuste, p_teste_calculadora_com_ajuste),
}

linhas_classificacao_calc = []
for nome_def, (y, p) in predicoes_calculadora.items():
    limiar_y = limiar_youden(y, p)
    limiares_a_testar = [('Youden', limiar_y)] + [(f'{l:.2f}', l) for l in LIMIARES_CLINICOS]

    for nome_limiar, limiar in limiares_a_testar:
        pred = (p >= limiar).astype(int)
        tn, fp, fn, tp = confusion_matrix(y, pred).ravel()
        sens = tp / (tp + fn) if (tp + fn) else np.nan
        espec = tn / (tn + fp) if (tn + fp) else np.nan
        vpp = tp / (tp + fp) if (tp + fp) else np.nan
        vpn = tn / (tn + fn) if (tn + fn) else np.nan
        linhas_classificacao_calc.append({
            'definicao': nome_def, 'limiar_nome': nome_limiar, 'limiar_valor': limiar,
            'VP': tp, 'FP': fp, 'VN': tn, 'FN': fn,
            'sensibilidade': sens, 'especificidade': espec, 'VPP': vpp, 'VPN': vpn,
            'accuracy': accuracy_score(y, pred), 'F1': f1_score(y, pred),
            'MCC': matthews_corrcoef(y, pred),
        })

tab_classificacao_calc = pd.DataFrame(linhas_classificacao_calc)
classif_calc_path = f'{OUTDIR}/fase6_calculadora_classificacao.csv'
tab_classificacao_calc.to_csv(classif_calc_path, index=False, encoding='utf-8-sig')
print(f'Classification table (calculator) saved: {classif_calc_path}')
print(tab_classificacao_calc.to_string(index=False))
print()
print('Note: accuracy and F1 are uninformative at this prevalence (~15% deaths) — MCC is the '
      'escalar preferencial (Chicco & Jurman, 2020).')

### Note — reading the table (calculator)

Expected consistency with Phase 5: the clinical thresholds (0.10 and 0.20) are the same, fixed, across
both definitions; the Youden threshold is recomputed on the calculator's own probability distribution in
each definition, and so may differ slightly from the full model's value. The practical reading of each
threshold (cost of the triggered action vs. cost of missing a true case) is the same as recorded in
Phase 5.

## 8 · Post-hoc recalibration — calculator

Same decision criterion and correction procedure as Phase 5 (Part I, Section 3), now applied to the
calculator, under both definitions:

1. **Decision**: 95% CI (bootstrap, 1,000 resamples) of the calibration intercept and slope on the test
   set. If the intercept's CI excludes 0, or the slope's excludes 1, there is evidence of systematic
   deviation.
2. **Fitting**: if indicated, Platt scaling is fitted on *out-of-fold* training predictions
   (`StratifiedGroupKFold`, 5 folds, grouped by patient) — never on the test set.
3. **Application**: the transformation is applied to the test predictions, and the Brier score
   before/after is compared.

In [ ]:
N_BOOT = 1000  # same value as Phase 5


def inclinacao_intercepto_calibracao(y, p, eps=1e-6):
    p_clip = np.clip(p, eps, 1 - eps)
    logit_p = np.log(p_clip / (1 - p_clip)).reshape(-1, 1)
    modelo_cal = LogisticRegression(penalty=None, solver='lbfgs')
    modelo_cal.fit(logit_p, y)
    return modelo_cal.intercept_[0], modelo_cal.coef_[0][0]


def bootstrap_ic_calibracao(y, p, n_boot=N_BOOT, seed=SEED):
    rng = np.random.default_rng(seed)
    n = len(y)
    interceptos, inclinacoes = [], []
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        if len(np.unique(y[idx])) < 2:
            continue
        i, s = inclinacao_intercepto_calibracao(y[idx], p[idx])
        interceptos.append(i); inclinacoes.append(s)
    ic_intercepto = (np.percentile(interceptos, 2.5), np.percentile(interceptos, 97.5))
    ic_inclinacao = (np.percentile(inclinacoes, 2.5), np.percentile(inclinacoes, 97.5))
    return ic_intercepto, ic_inclinacao


linhas_decisao_recal_calc = []
for nome_def, (y, p) in predicoes_calculadora.items():
    intercepto, inclinacao = inclinacao_intercepto_calibracao(y, p)
    ic_intercepto, ic_inclinacao = bootstrap_ic_calibracao(y, p)
    recalibrar = (ic_intercepto[0] > 0 or ic_intercepto[1] < 0) or (ic_inclinacao[0] > 1 or ic_inclinacao[1] < 1)
    linhas_decisao_recal_calc.append({
        'definicao': nome_def, 'intercepto': intercepto, 'inclinacao': inclinacao,
        'IC_intercepto_low': ic_intercepto[0], 'IC_intercepto_high': ic_intercepto[1],
        'IC_inclinacao_low': ic_inclinacao[0], 'IC_inclinacao_high': ic_inclinacao[1],
        'recalibrar': recalibrar,
    })
    print(f"{nome_def}: intercepto={intercepto:.3f} IC95%[{ic_intercepto[0]:.3f}, {ic_intercepto[1]:.3f}] | "
          f"slope={inclinacao:.3f} 95%CI[{ic_inclinacao[0]:.3f}, {ic_inclinacao[1]:.3f}] | "
          f"recalibrar = {recalibrar}")

tab_decisao_recal_calc = pd.DataFrame(linhas_decisao_recal_calc)

In [ ]:
def obter_predicoes_out_of_fold(modelo_ajustado, Xtr, ytr, groups_tr, cats, n_splits=5, seed=SEED):
    # Out-of-fold predictions on training, refitting a clone of the model (same hyperparameters) on
    # each fold of the StratifiedGroupKFold — never using the test set itself.
    cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    p_oof = np.full(len(ytr), np.nan)
    is_lightgbm = 'LGBM' in type(modelo_ajustado).__name__

    for idx_treino_fold, idx_holdout_fold in cv.split(Xtr, ytr, groups=groups_tr):
        modelo_fold = clone(modelo_ajustado)
        X_tr_fold = Xtr.iloc[idx_treino_fold]
        X_ho_fold = Xtr.iloc[idx_holdout_fold]
        y_tr_fold = ytr[idx_treino_fold]

        if is_lightgbm:
            modelo_fold.fit(X_tr_fold, y_tr_fold, categorical_feature=cats)
        else:
            modelo_fold.fit(X_tr_fold, y_tr_fold)

        p_oof[idx_holdout_fold] = modelo_fold.predict_proba(X_ho_fold)[:, 1]

    return p_oof


cats_calc_sem_ajuste = [c for c in cats_sem_ajuste if c in variaveis_top_sem_ajuste]
cats_calc_com_ajuste = [c for c in cats_com_ajuste if c in variaveis_top_com_ajuste]

dados_calibracao_calc = {
    'Sem ajuste': {
        'modelo': calculadora_sem_ajuste, 'Xtr': Xtr_sem_ajuste[variaveis_top_sem_ajuste],
        'ytr': ytr_sem_ajuste, 'groups_tr': treino_sem_ajuste['patient_id'].values,
        'cats': cats_calc_sem_ajuste, 'p_teste': p_teste_calculadora_sem_ajuste, 'y_teste': yte_sem_ajuste,
    },
    'Com ajuste': {
        'modelo': calculadora_com_ajuste, 'Xtr': Xtr_com_ajuste[variaveis_top_com_ajuste],
        'ytr': ytr_com_ajuste, 'groups_tr': groups_tr_com_ajuste,
        'cats': cats_calc_com_ajuste, 'p_teste': p_teste_calculadora_com_ajuste, 'y_teste': yte_com_ajuste,
    },
}

linhas_recalibracao_calc = []
predicoes_recalibradas_calc = {}
for nome_def, dados in dados_calibracao_calc.items():
    linha_decisao = tab_decisao_recal_calc[tab_decisao_recal_calc['definicao'] == nome_def].iloc[0]
    p_teste, y_teste = dados['p_teste'], dados['y_teste']

    if not linha_decisao['recalibrar']:
        print(f'{nome_def}: 95% CI compatible with adequate calibration — recalibration not applied.')
        predicoes_recalibradas_calc[nome_def] = p_teste
        continue

    p_oof = obter_predicoes_out_of_fold(dados['modelo'], dados['Xtr'], dados['ytr'], dados['groups_tr'], dados['cats'])
    intercepto_platt, inclinacao_platt = inclinacao_intercepto_calibracao(dados['ytr'], p_oof)

    logit_teste = np.log(np.clip(p_teste, 1e-6, 1 - 1e-6) / (1 - np.clip(p_teste, 1e-6, 1 - 1e-6)))
    logit_recal = intercepto_platt + inclinacao_platt * logit_teste
    p_recal = 1 / (1 + np.exp(-logit_recal))
    predicoes_recalibradas_calc[nome_def] = p_recal

    brier_antes = brier_score_loss(y_teste, p_teste)
    brier_depois = brier_score_loss(y_teste, p_recal)
    intercepto_pos, inclinacao_pos = inclinacao_intercepto_calibracao(y_teste, p_recal)

    linhas_recalibracao_calc.append({
        'definicao': nome_def, 'intercepto_platt_oof': intercepto_platt,
        'inclinacao_platt_oof': inclinacao_platt, 'Brier_antes': brier_antes, 'Brier_depois': brier_depois,
        'intercepto_pos_recal': intercepto_pos, 'inclinacao_pos_recal': inclinacao_pos,
    })
    print(f'{nome_def}: recalibrado (Platt out-of-fold). Brier {brier_antes:.4f} -> {brier_depois:.4f}')

tab_recalibracao_calc = pd.DataFrame(linhas_recalibracao_calc)
recal_calc_path = f'{OUTDIR}/fase6_calculadora_recalibracao.csv'
decisao_calc_path = f'{OUTDIR}/fase6_calculadora_decisao_recalibracao.csv'
tab_recalibracao_calc.to_csv(recal_calc_path, index=False, encoding='utf-8-sig')
tab_decisao_recal_calc.to_csv(decisao_calc_path, index=False, encoding='utf-8-sig')
print(f'Tabelas salvas: {recal_calc_path} | {decisao_calc_path}')

In [ ]:
# --- Plot: bootstrap distribution of the intercept (decision) + calibration curve before/after, calculator ---
fig, axes = plt.subplots(2, len(predicoes_calculadora), figsize=(7 * len(predicoes_calculadora), 10))

rng_plot = np.random.default_rng(SEED)

for col_idx, (nome_def, (y_teste, p_teste)) in enumerate(predicoes_calculadora.items()):
    linha_decisao = tab_decisao_recal_calc[tab_decisao_recal_calc['definicao'] == nome_def].iloc[0]

    n = len(y_teste)
    interceptos_boot = []
    for _ in range(N_BOOT):
        idx = rng_plot.integers(0, n, n)
        if len(np.unique(y_teste[idx])) < 2:
            continue
        i_boot, _ = inclinacao_intercepto_calibracao(y_teste[idx], p_teste[idx])
        interceptos_boot.append(i_boot)

    eixo_top = axes[0, col_idx]
    eixo_top.hist(interceptos_boot, bins=40, color='#D9D9D9', edgecolor='white', zorder=1)
    eixo_top.axvline(0, color='gray', linestyle=':', linewidth=2,
                      label='Reference (perfect calibration) = 0', zorder=2)
    eixo_top.axvline(linha_decisao['intercepto'], color='#C00000', linestyle='-', linewidth=2.5,
                      label=f"Intercepto observado = {linha_decisao['intercepto']:.3f}", zorder=3)
    eixo_top.axvline(linha_decisao['IC_intercepto_low'], color='#1F3864', linestyle='--', linewidth=1.5, zorder=2)
    eixo_top.axvline(linha_decisao['IC_intercepto_high'], color='#1F3864', linestyle='--', linewidth=1.5,
                      label='IC 95% (bootstrap)', zorder=2)
    eixo_top.set_title(f'Calculadora — {nome_def}\nRecalibrar = {linha_decisao["recalibrar"]}')
    eixo_top.set_xlabel('Calibration intercept (bootstrap)')
    eixo_top.legend(fontsize=8, loc='upper left')

    p_recal = predicoes_recalibradas_calc[nome_def]
    identicas = np.allclose(p_teste, p_recal)

    frac_antes, media_antes = calibration_curve(y_teste, p_teste, n_bins=10, strategy='quantile')
    frac_depois, media_depois = calibration_curve(y_teste, p_recal, n_bins=10, strategy='quantile')

    eixo_bot = axes[1, col_idx]
    eixo_bot.plot(media_antes, frac_antes, 'o-', color='#C00000', linewidth=2.5, markersize=8,
                  label='Before recalibration', zorder=2)
    eixo_bot.plot(media_depois, frac_depois, 's--', color='#1F3864', linewidth=1.5, markersize=5,
                  label='After recalibration', zorder=3)
    eixo_bot.plot([0, 1], [0, 1], '--', color='gray', linewidth=1, zorder=1)
    eixo_bot.set_xlabel('Probabilidade prevista')
    eixo_bot.set_ylabel('Observed frequency')
    if identicas:
        eixo_bot.text(0.05, 0.90, 'Identical curves — recalibration not applied',
                      transform=eixo_bot.transAxes, fontsize=8, color='#C00000')
    eixo_bot.legend(fontsize=8, loc='lower right')

plt.tight_layout()
fig_calib_calc_path = f'{OUTDIR}/fase6_calculadora_calibracao.png'
fig.savefig(fig_calib_calc_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Figura salva: {fig_calib_calc_path}')

## 9 · Package the calculator as a function

One function per definition, receiving a dictionary (or a one-row DataFrame) with the N (30) variables
and returning the predicted probability of hospital death. Both use the same engine (XGBoost), just
fitted on cohorts with differently defined outcomes.

In [ ]:
def construir_pacote_calculadora(modelo, variaveis, cats, Xtr_referencia):
    # Stores, for each categorical column, the exact list of categories seen in training —
    # needed so a new input is encoded the same way the model learned.
    categorias_por_coluna = {c: Xtr_referencia[c].cat.categories for c in cats if c in Xtr_referencia.columns}
    return {'modelo': modelo, 'variaveis': variaveis, 'categoricas': cats, 'categorias_treino': categorias_por_coluna}


def calculadora_risco(dados_paciente, pacote):
    # patient_data: dict or 1-row DataFrame, with the keys/columns in package['variaveis'].
    if isinstance(dados_paciente, dict):
        dados_paciente = pd.DataFrame([dados_paciente])
    X = dados_paciente[pacote['variaveis']].copy()
    for c in pacote['categoricas']:
        if c in X.columns:
            categorias = pacote['categorias_treino'][c]
            X[c] = pd.Categorical(X[c].astype('string'), categories=categorias)
    return float(pacote['modelo'].predict_proba(X)[:, 1][0])


cats_calc_sem_ajuste = [c for c in cats_sem_ajuste if c in variaveis_top_sem_ajuste]
cats_calc_com_ajuste = [c for c in cats_com_ajuste if c in variaveis_top_com_ajuste]

pacote_sem_ajuste = construir_pacote_calculadora(calculadora_sem_ajuste, variaveis_top_sem_ajuste, cats_calc_sem_ajuste, Xtr_sem_ajuste)
pacote_com_ajuste = construir_pacote_calculadora(calculadora_com_ajuste, variaveis_top_com_ajuste, cats_calc_com_ajuste, Xtr_com_ajuste)

# Usage example, with the first test patient:
exemplo = Xte_sem_ajuste[variaveis_top_sem_ajuste].iloc[[0]]
p_exemplo = calculadora_risco(exemplo.to_dict('records')[0], pacote_sem_ajuste)
print(f'Example — predicted probability (calculator, unadjusted): {p_exemplo:.4f}')

joblib.dump(pacote_sem_ajuste, f'{OUTDIR}/calculadora_risco_sem_ajuste.joblib')
joblib.dump(pacote_com_ajuste, f'{OUTDIR}/calculadora_risco_com_ajuste.joblib')
print('Calculators saved (unadjusted and adjusted).')

## 10 · TRIPOD-AI checklist

Mapping of the main TRIPOD-AI items (Collins et al., 2024) to what has already been produced in each
phase — to guide the manuscript's writing, not as a substitute for it.

In [ ]:
checklist_tripod_ai = pd.DataFrame([
    {'secao': 'Title/Abstract', 'item': 'Identification as a prediction study, target population, outcome', 'evidencia_fase': 'Fase 1', 'status': 'Completed'},
    {'secao': 'Introduction', 'item': 'Rationale and objectives (comparison with SAPS3)', 'evidencia_fase': 'Arquitetura do estudo', 'status': 'Completed'},
    {'secao': 'Methods', 'item': 'Fonte de dados e elegibilidade', 'evidencia_fase': 'Fase 1', 'status': 'Completed'},
    {'secao': 'Methods', 'item': 'Outcome definition and ascertainment moment', 'evidencia_fase': 'Phase 1 (+ transfer-bias finding)', 'status': 'Completed, with documented limitation'},
    {'secao': 'Methods', 'item': 'Preditores candidatos e momento de coleta (janela de 6h)', 'evidencia_fase': 'Fase 2', 'status': 'Completed'},
    {'secao': 'Methods', 'item': 'Tamanho amostral', 'evidencia_fase': 'Phase 1 (5,676 index admissions)', 'status': 'Completed'},
    {'secao': 'Methods', 'item': 'Tratamento de dados ausentes', 'evidencia_fase': 'Phase 2 (missingness indicators; native in the boosting engines)', 'status': 'Completed'},
    {'secao': 'Methods', 'item': 'Statistical/ML methods and model comparison', 'evidencia_fase': 'Fases 3-4', 'status': 'Completed'},
    {'secao': 'Methods', 'item': 'Validation (internal, temporal strategy)', 'evidencia_fase': 'Fases 3-4', 'status': 'Completed'},
    {'secao': 'Methods', 'item': 'Performance metrics (discrimination, calibration, clinical utility)', 'evidencia_fase': 'Fases 4-5', 'status': 'Completed'},
    {'secao': 'Methods', 'item': 'Reamostragem/desbalanceamento', 'evidencia_fase': 'Phase 4 (no resampling; no scale_pos_weight)', 'status': 'Completed'},
    {'secao': 'Methods', 'item': 'Interpretability (variable importance)', 'evidencia_fase': 'Fases 4-5 (SHAP global e local)', 'status': 'Completed'},
    {'secao': 'Resultados', 'item': 'Participant flow and characteristics (Table 1)', 'evidencia_fase': 'Fase 2', 'status': 'Completed'},
    {'secao': 'Resultados', 'item': 'Desempenho do modelo, com IC', 'evidencia_fase': 'Fases 4-5', 'status': 'Completed'},
    {'secao': 'Resultados', 'item': 'Comparison with clinical reference model (SAPS3)', 'evidencia_fase': 'Fases 3-4', 'status': 'Completed'},
    {'secao': 'Resultados', 'item': 'Subgroup/fairness analysis', 'evidencia_fase': 'Fase 5', 'status': 'Completed'},
    {'secao': 'Resultados', 'item': 'Simplified model (if applicable)', 'evidencia_fase': 'Fase 6', 'status': 'Completed'},
    {'secao': 'Discussion', 'item': 'Limitations (incl. transfer bias, lack of external validation)', 'evidencia_fase': 'Transfer-bias note + this phase', 'status': 'Completed'},
    {'secao': 'Discussion', 'item': 'Implications for clinical use', 'evidencia_fase': 'Pending write-up', 'status': 'Pendente'},
    {'secao': 'Other information', 'item': 'Code/model availability', 'evidencia_fase': 'Notebooks e .joblib salvos', 'status': 'Completed'},
    {'secao': 'Other information', 'item': 'Financiamento e conflitos de interesse', 'evidencia_fase': 'Pending write-up', 'status': 'Pendente'},
])

checklist_path = f'{OUTDIR}/fase6_checklist_tripod_ai.csv'
checklist_tripod_ai.to_csv(checklist_path, index=False, encoding='utf-8-sig')
print(f'Checklist salvo: {checklist_path}')
print(checklist_tripod_ai.to_string(index=False))

## 11 · PROBAST assessment (risk of bias)

Judgment by domain, with the transfer-bias finding explicitly recorded in the Outcome domain — PROBAST
asks exactly for this kind of explicit assessment, not a side note.

In [ ]:
avaliacao_probast = pd.DataFrame([
    {'dominio': '1. Participants', 'pergunta_chave': 'Are the source and eligibility criteria appropriate?',
     'julgamento': 'Low risk', 'justificativa': 'Index-admission cohort, with validated patient identity resolution (connected components) and recovery of ambiguous pairs by positional alignment.'},
    {'dominio': '2. Predictors', 'pergunta_chave': 'Are predictors defined and measured consistently, without post-outcome information?',
     'julgamento': 'Low risk', 'justificativa': 'Predictors restricted to the 6-h admission window; the discharge-destination variable (post-outcome) correctly excluded from the predictor list.'},
    {'dominio': '3. Outcome', 'pergunta_chave': 'Was the outcome determined correctly and without systematic differences across predictor subgroups?',
     'julgamento': 'Uncertain risk — bias identified and quantified', 'justificativa': "Hospital mortality derived from HospitalDischargeName (binary discharge/death) counts patients transferred to another hospital as survivors. Confirmed empirically: Cardiovascular mortality rises from 6.4% to 8.2% (+28.0%) under the adjusted definition. Set 2's advantage over SAPS3 holds under both definitions (p<0.001 unadjusted and p<0.001 adjusted) — the bias does not compromise this specific comparison, but outcome classification itself remains uncertain for transferred patients, whose real status is unknown. Not corrected in the unadjusted definition; quantified and documented in this phase."},
    {'dominio': '4. Analysis', 'pergunta_chave': 'Is the sample size adequate, is missing-data handling appropriate, and does validation avoid leakage?',
     'julgamento': 'Low risk', 'justificativa': 'Temporal (primary) and random-by-patient (reference) validation; StratifiedGroupKFold restricted to training; no resampling that distorts calibration; fine calibration and recalibration decision by a pre-specified statistical criterion (bootstrap).'},
])

avaliacao_path = f'{OUTDIR}/fase6_avaliacao_probast.csv'
avaliacao_probast.to_csv(avaliacao_path, index=False, encoding='utf-8-sig')
print(f'PROBAST assessment saved: {avaliacao_path}')
print(avaliacao_probast.to_string(index=False))

---
### What this phase delivered
- Set 2's full model under both outcome definitions (unadjusted and adjusted), side by side.
- SHAP importance ranking recomputed separately for each definition, with a comparison of the two top-N
  (30) variable lists.
- Risk calculator (restricted model, N=30) fitted and validated (ROC, DeLong) for each definition.
- Full validation of the calculator, replicating Phase 5's methodology: classification metrics at three
  thresholds (Youden, ≈0.10, ≈0.20; MCC as the preferred scalar summary) and post-hoc recalibration
  (bootstrap decision + out-of-fold Platt scaling, with before/after Brier), for both definitions.
- Calculators packaged as Python functions, saved as `.joblib` for reuse.
- TRIPOD-AI checklist filled in, indicating what is complete and what remains for the final write-up.
- PROBAST assessment by domain, with transfer bias explicitly recorded in the Outcome domain as an
  uncertain risk — not low risk — until a final decision on which definition to adopt is made.